# Final Algorithm to Train Combined Leakage Model

Based on Exploration_leakage_combined.ipynb files (Explore and Edit there, put the final code here)

In [ ]:
# Dependencies

import numpy as np
import pandas as pd
import re
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## Load and prepare reference (model) data and Monorail data (one or more kits)

align common columns, and return a single combined DataFrame.

In [ ]:
# ======================================================================
# STEP 1: LOAD AND PREPARE DATA (REFERENCE + MULTI-KIT MONORAIL)
# ======================================================================

def load_data(model_path, monorail_paths):
    """
    Load and prepare reference (model) data and Monorail data (one or more kits),
    align common columns, and return a single combined DataFrame.

    Parameters
    ----------
    model_path : str
        Path to model.csv (reference experimental campaign).
    monorail_paths : str or list of str
        Path or list of paths to Monorail TestBrakefinal_data_kitXX.csv files.

    Returns
    -------
    df_base : pandas.DataFrame
        Combined DataFrame with:
        - aligned common columns between reference and Monorail,
        - binary label (0/1) where available,
        - 'Source' column (kit ID or 0 for reference),
        - 'DataSource' column (0 = reference, 1 = Monorail).
    """

    # --------------------------------------------------------------
    # Helper: load and clean ONE Monorail file
    # --------------------------------------------------------------
    def load_Monorail(filepath: str) -> pd.DataFrame:
        df = pd.read_csv(filepath)

        # Keep only standard braking
        if 'Non_Standard_Braking' in df.columns:
            df = df[df['Non_Standard_Braking'] == 0]

        # Extract numeric kit ID from filename, e.g. "TestBrakefinal_data_kit06.csv" -> 6
        match = re.search(r'kit(\d+)', os.path.basename(filepath))
        source = int(match.group(1)) if match else -1
        df['Source'] = source

        # Convert "xx sec" string columns to float seconds where possible
        for col in df.select_dtypes(include='object'):
            try:
                df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
            except (AttributeError, ValueError):
                # AttributeError if column is not string-like; ValueError if some values cannot be cast
                continue

        return df

    # --------------------------------------------------------------
    # 1) REFERENCE DATA: load, label, aggregate
    # --------------------------------------------------------------
    df_reference = pd.read_csv(model_path)
    df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)

    # Binary label from malfunction code
    leakage_codes = ['C', 'D', 'E', 'F', 'G']
    df_reference['LeakageLabel'] = np.where(
        df_reference['Malfunction'].isin(leakage_codes),
        'Combined leakage',
        'Healthy'
    )

    # Add Source = 0 for reference campaign
    df_reference['Source'] = 0

    # Aggregate delay and efficiency columns
    delay_eff_map = {
        'Total_timing_delay':      ['Brake_timing_delay_exp',      'Release_timing_delay_exp'],
        'Total_energy_delay':      ['Brake_energy_delay_exp',      'Release_energy_delay_exp'],
        'Total_power_delay':       ['Brake_power_delay_exp',       'Release_power_delay_exp'],
        'Total_power_efficiency':  ['Brake_power_efficiency_exp',  'Release_power_efficiency_exp'],
        'Total_energy_efficiency': ['Brake_energy_effiency_exp',   'Release_energy_efficiency_exp']
    }

    for new_col, (c1, c2) in delay_eff_map.items():
        # If any of these columns are missing in some version of model.csv, guard with .get
        if c1 in df_reference.columns and c2 in df_reference.columns:
            df_reference[new_col] = df_reference[c1] + df_reference[c2]

    # Drop original per-phase columns (only those that actually exist)
    cols_to_drop = [c for pair in delay_eff_map.values() for c in pair if c in df_reference.columns]
    df_reference.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # Rename to your canonical names
    rename_map = {
        'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
        'Buildup_end_pressure_delay_exp':  'Buildup_end_pressure_delay',
        'Weight':                          'WV_MeanPressure',
        'Brake_action':                    'EmergencyBrake_action'
    }
    df_reference.rename(columns=rename_map, inplace=True)

    # --------------------------------------------------------------
    # 2) MONORAIL DATA: load one or more kit files
    # --------------------------------------------------------------
    if isinstance(monorail_paths, str):
        monorail_paths = [monorail_paths]

    dfs_mono = [load_Monorail(fp) for fp in monorail_paths]
    df_data = pd.concat(dfs_mono, ignore_index=True)

    # --------------------------------------------------------------
    # 3) ALIGN STRUCTURES AND COMBINE
    # --------------------------------------------------------------
    # Ensure 'Source' is integer in both
    df_reference['Source'] = df_reference['Source'].astype(int)
    df_data['Source']      = df_data['Source'].astype(int)

    # Columns common to BOTH datasets
    common_cols = df_reference.columns.intersection(df_data.columns).tolist()

    # Subsets with only common columns + a DataSource flag
    df_reference_subset = df_reference[common_cols].copy()
    df_reference_subset['DataSource'] = 0  # 0 = reference campaign

    df_data_subset = df_data[common_cols].copy()
    df_data_subset['DataSource'] = 1       # 1 = Monorail (real-time) data

    # Stack reference + Monorail
    df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)

    # Encode final label column (will be NaN for Monorail if it has no LeakageLabel)
    if 'LeakageLabel' in df_combined.columns:
        df_combined.rename(columns={'LeakageLabel': 'label'}, inplace=True)
        df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'Combined leakage': 1})

    # Convert any remaining "xx sec" string columns to float (esp. from model.csv)
    for col in df_combined.select_dtypes(include='object'):
        try:
            df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
        except (AttributeError, ValueError):
            continue

    df_base = df_combined.copy()
    return df_base

model_path = 'model.csv'
monorail_paths = [
    'TestBrakefinal_data_kit01.csv',
    'TestBrakefinal_data_kit06.csv',
    'TestBrakefinal_data_kit27.csv'
]

df_base = load_data(model_path, monorail_paths)

print(df_base.shape)
print(df_base['DataSource'].value_counts(dropna=False))
print(df_base['label'].value_counts(dropna=False))  # will include NaN for unlabeled Monorail


## Data Preprocessing, Split data into Train/Test

In [ ]:
# ============================================================================
# STEP 2: DATA PREPROCESSING
# ============================================================================

def preprocess_data(df, test_size=0.2):
    """
    Preprocess data: split into train/test and separate healthy from leakage
    """
    
    mask = (df["WV_MeanPressure"] >= 2) & (df["WV_MeanPressure"] <= 3)
    df_filt = df[mask].copy()
    
    # Separate features and labels
    # X = df_filt.drop('label', axis=1)
    X = df_filt[['Total_power_efficiency']]
    y = df_filt['label']
    
    # Split data (stratified to preserve class ratio)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )
    
    # Extract only HEALTHY samples for training (for anomaly detection models)
    X_train_healthy = X_train[y_train == 0]
    
    print(f"Total samples: {len(df)}")
    print(f"Training samples: {len(X_train)} (Healthy: {sum(y_train==0)}, Leakage: {sum(y_train==1)})")
    print(f"Training samples (healthy only): {len(X_train_healthy)}")
    print(f"Test samples: {len(X_test)} (Healthy: {sum(y_test==0)}, Leakage: {sum(y_test==1)})")
    
    return X_train, X_test, y_train, y_test, X_train_healthy

[X_train, X_test, y_train, y_test, X_train_healthy] = preprocess_data(df_base, test_size=0.2)

X_train.head()

In [ ]:
# ============================================================================
# STEP 3: IMPUTE + FEATURE SCALING
# ============================================================================

from sklearn.impute import SimpleImputer

def scale_features(X_train, X_test, X_train_healthy):
    """
    Impute missing values by median, then standardize features.
    Returns imputed+scaled arrays, plus fitted scaler and imputer.
    """
    # 1) Median imputation (fit only on training set)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)
    X_train_healthy_imp = imputer.transform(X_train_healthy)

    # 2) Standardization (fit only on imputed training set)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    X_train_healthy_scaled = scaler.transform(X_train_healthy_imp)

    return X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer

[X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer] = scale_features(X_train, X_test, X_train_healthy)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score
)

# for SMOTE & its pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


# ============================================================================
# STEP 4: DEFINE ALL MODELS
# ============================================================================

def get_all_models(contamination=0.05):
    """
    Define all models to be tested
    Returns dictionary of models grouped by type
    """
    models = {
        # === ANOMALY DETECTION MODELS (train on healthy data only) ===
        'anomaly': {
            'Isolation Forest': IsolationForest(
                contamination=contamination,
                random_state=42,
                n_estimators=200,
                n_jobs=-1
            ),
            'One-Class SVM': OneClassSVM(
                nu=contamination,
                kernel='rbf',
                gamma='auto'
            ),
            'Local Outlier Factor': LocalOutlierFactor(
                contamination=contamination,
                novelty=True,
                n_neighbors=25
            )
        },
        
        # === SUPERVISED MODELS (train on imbalanced data) ===
        'supervised_imbalanced': {
            'Random Forest (Weighted)': RandomForestClassifier(
                class_weight='balanced',
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            ),
            'XGBoost (Weighted)': XGBClassifier(
                scale_pos_weight=(1100/23),  # ratio of negative/positive
                n_estimators=100,
                random_state=42,
                eval_metric='logloss'
            ),
            'Logistic Regression (Weighted)': LogisticRegression(
                class_weight='balanced',
                random_state=42,
                max_iter=1000
            ),
            # NEW: plain decision tree with class weights
            'Decision Tree (Weighted)': DecisionTreeClassifier(
                class_weight='balanced',
                random_state=42,
                max_depth=3  # you can tune this
            ),
            
        },
        
        # === SUPERVISED MODELS WITH SMOTE (train on balanced data) ===
        'supervised_smote': {
            'Random Forest (SMOTE)': RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            ),
            'XGBoost (SMOTE)': XGBClassifier(
                n_estimators=200,
                random_state=42,
                eval_metric='logloss'
            ),
            'Decision Tree (SMOTE)': DecisionTreeClassifier(
                random_state=42,
                max_depth=3
            )
        }
    }
    
    return models

# ============================================================================
# STEP 5: TRAIN ALL MODELS
# ============================================================================

def train_all_models(X_train_scaled, y_train, X_train_healthy_scaled, contamination=0.05):
    """
    Train all models and return trained models with metadata
    """
    models = get_all_models(contamination)
    trained_models = {}
    
    print("\n" + "="*60)
    print("TRAINING ALL MODELS")
    print("="*60)
    
    # Train anomaly detection models (on healthy data only)
    print("\n[1/3] Training Anomaly Detection Models...")
    for name, model in models['anomaly'].items():
        print(f"  - Training {name}...")
        model.fit(X_train_healthy_scaled)
        trained_models[name] = {
            'model': model,
            'type': 'anomaly',
            'trained': True
        }
    
    # Train supervised models on imbalanced data
    print("\n[2/3] Training Supervised Models (Imbalanced Data)...")
    for name, model in models['supervised_imbalanced'].items():
        print(f"  - Training {name}...")
        model.fit(X_train_scaled, y_train)
        trained_models[name] = {
            'model': model,
            'type': 'supervised',
            'trained': True
        }
    
    # Apply SMOTE and train supervised models
    print("\n[3/3] Training Supervised Models (with SMOTE)...")
    smote = SMOTE(random_state=42)
    X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
    print(f"  - After SMOTE: {sum(y_train_smote==0)} healthy, {sum(y_train_smote==1)} leakage")
    
    for name, model in models['supervised_smote'].items():
        print(f"  - Training {name}...")
        model.fit(X_train_smote, y_train_smote)
        trained_models[name] = {
            'model': model,
            'type': 'supervised_smote',
            'trained': True
        }
    
    print("\nAll models trained successfully!")
    return trained_models

def cross_validate_all_models_with_metrics(trained_models, X, y, k=5, out_csv=None):
    """
    Cross-validate all models with multiple metrics and return
    a ranked pandas DataFrame. Optionally saves to CSV.
    """

    # --- ensure NumPy arrays to avoid pandas indexing issues ---
    X = np.asarray(X)
    y = np.asarray(y)

    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    records = []

    print("\n" + "="*80)
    print(f"RUNNING {k}-FOLD CROSS-VALIDATION FOR ALL MODELS (MULTI-METRIC)")
    print("="*80)

    for name, entry in trained_models.items():
        base_model = entry['model']
        mtype = entry['type']

        print(f"\n→ Evaluating {name}  (type = {mtype})")

        f1_list, prec_list, rec_list = [], [], []
        roc_list, pr_list = [], []  # ROC-AUC, PR-AUC

        for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            # --- select proper model per type ---
            if mtype == 'supervised':
                model = clone(base_model)
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)

                score_vals = None
                if hasattr(model, "predict_proba"):
                    score_vals = model.predict_proba(X_test)[:, 1]
                elif hasattr(model, "decision_function"):
                    score_vals = model.decision_function(X_test)

            elif mtype == 'supervised_smote':
                # SMOTE must be inside the CV loop to avoid leakage
                model = ImbPipeline([
                    ('smote', SMOTE(random_state=42)),
                    ('clf', clone(base_model))
                ])
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)

                score_vals = None
                clf = model.named_steps['clf']
                if hasattr(clf, "predict_proba"):
                    score_vals = clf.predict_proba(X_test)[:, 1]
                elif hasattr(clf, "decision_function"):
                    score_vals = clf.decision_function(X_test)


            elif mtype == 'anomaly':
                # Train only on healthy samples in the training fold
                X_train_healthy = X_train[y_train == 0]
                model = clone(base_model)
                model.fit(X_train_healthy)

                raw_pred = model.predict(X_test)          # -1 = anomaly, 1 = normal
                y_pred = (raw_pred == -1).astype(int)     # 1 = leakage
                score_vals = None  # no continuous scores here

            else:
                raise ValueError(f"Unknown model type: {mtype}")

            # --- basic classification metrics ---
            f1_list.append(f1_score(y_test, y_pred, zero_division=0))
            prec_list.append(precision_score(y_test, y_pred, zero_division=0))
            rec_list.append(recall_score(y_test, y_pred, zero_division=0))

            # --- ROC-AUC and PR-AUC (only if we have scores and both classes present) ---
            if score_vals is not None and len(np.unique(y_test)) == 2:
                try:
                    roc_list.append(roc_auc_score(y_test, score_vals))
                    pr_list.append(average_precision_score(y_test, score_vals))
                except ValueError:
                    # can still fail if only 1 class in y_test after all
                    pass

        # --- aggregate results for this model ---
        record = {
            'Model': name,
            'Type': mtype,
            'F1_mean':   np.mean(f1_list),
            'F1_std':    np.std(f1_list),
            'Precision_mean': np.mean(prec_list),
            'Recall_mean':    np.mean(rec_list),
            'ROC_AUC_mean':  np.mean(roc_list) if len(roc_list) > 0 else np.nan,
            'PR_AUC_mean':   np.mean(pr_list)  if len(pr_list) > 0 else np.nan,
        }
        records.append(record)

        print(f"  F1      per-fold: {np.round(f1_list, 3)}  | mean = {record['F1_mean']:.3f}")
        print(f"  Prec    per-fold: {np.round(prec_list, 3)} | mean = {record['Precision_mean']:.3f}")
        print(f"  Recall  per-fold: {np.round(rec_list, 3)} | mean = {record['Recall_mean']:.3f}")
        if not np.isnan(record['ROC_AUC_mean']):
            print(f"  ROC-AUC mean   = {record['ROC_AUC_mean']:.3f}")
        if not np.isnan(record['PR_AUC_mean']):
            print(f"  PR-AUC  mean   = {record['PR_AUC_mean']:.3f}")

    # --- build table and rank by mean F1 ---
    df_results = pd.DataFrame(records)
    df_results_sorted = df_results.sort_values(by='F1_mean', ascending=False).reset_index(drop=True)

    print("\n" + "="*80)
    print("CROSS-VALIDATION SUMMARY (RANKED BY MEAN F1)")
    print("="*80)
    print(df_results_sorted)

    if out_csv is not None:
        df_results_sorted.to_csv(out_csv, index=False)
        print(f"\nSaved CV summary to: {out_csv}")

    return df_results_sorted



trained_models = train_all_models(X_train_scaled, y_train, X_train_healthy_scaled, contamination=0.05)

cv_summary = cross_validate_all_models_with_metrics(
    trained_models,
    X_train_scaled,
    y_train,
    k=5,
    out_csv="cv_results_summary.csv"
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Optional: to make plots look nicer
plt.style.use('seaborn-v0_8')  # or comment out if you prefer default

# --- Prepare data ---
models = cv_summary['Model'].values
f1_mean = cv_summary['F1_mean'].values
f1_std  = cv_summary['F1_std'].values
types   = cv_summary['Type'].values  # 'supervised', 'supervised_smote', 'anomaly'

# Assign a color per type
type_to_color = {
    'supervised': '#1f77b4',        # blue
    'supervised_smote': '#2ca02c',  # green
    'anomaly': '#d62728'            # red
}
colors = [type_to_color[t] for t in types]

# --- Create bar chart ---
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(models))

bars = ax.bar(x, f1_mean, yerr=f1_std, capsize=5, color=colors, alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha='right')
ax.set_ylabel('F1-score (mean ± std)')
ax.set_title('Cross-validated F1-score per Model')

# Build custom legend
handles = []
labels  = []
for t, c in type_to_color.items():
    handles.append(plt.Rectangle((0, 0), 1, 1, color=c))
    labels.append(t)
ax.legend(handles, labels, title='Model Type')

ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()
